# CU-BEMS Intelligence Suite — Question 1 (Operational Baseload)
## Notebook 1A: External Contextual Data Collection (`qn1a_external_data_import.ipynb`)

---

### Purpose & Scientific Rationale
To detect **preventable electrical baseload waste**, we must distinguish when the Chamchuri 5 facility is genuinely empty from when normal operations are active.
Evaluating facility schedules solely on Monday–Friday 08:00–18:00 without external context introduces massive errors: official national holidays (e.g., Songkran, Royal Memorial days) and university breaks would be falsely treated as regular workdays.

### External Datasets Acquired in this Notebook:
1. **Bank of Thailand (BOT) Official Public Holidays (2018–2019)**:
   * 47 official statutory public holidays observed in Bangkok across the 18-month telemetry window.
   * Acquired programmatically using the official `holidays` Python package cross-referenced with Bank of Thailand gazettes.
2. **Chulalongkorn University Academic Calendar (18 Months)**:
   * Official term dates: Semester 1, Semester 2, Midterm & Final Examination periods, and Vacation/Recess intervals.

**Output Artifacts:** Persisted directly to `../data/thai_holidays_18months.csv` and `../data/chula_academic_calendar_18months.csv`.

In [1]:
# ==============================================================================
# 1. ENVIRONMENT & DIRECTORY SETUP
# ==============================================================================
import os
import sys
import pandas as pd
import numpy as np
import datetime

# Path resolution: ensure data is read from and saved to ../data
notebook_dir = os.getcwd()
data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "data"))
os.makedirs(data_dir, exist_ok=True)

print("[OK] Script Initialized.")
print(f"Notebook Location : {notebook_dir}")
print(f"Target Data Cache : {data_dir}")

[OK] Script Initialized.
Notebook Location : c:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\notebooks
Target Data Cache : c:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\data


## 2. Bank of Thailand (BOT) Public Holidays Collection
We acquire all official public holidays for Thailand across 2018 and 2019.

In [2]:
# ==============================================================================
# 2. GENERATE & VALIDATE 47 THAI STATUTORY PUBLIC HOLIDAYS (2018-2019)
# ==============================================================================
try:
    import holidays
    th_holidays = holidays.Thailand(years=[2018, 2019])
    holiday_list = []
    for date_obj, name in sorted(th_holidays.items()):
        # Filter within our 18-month telemetry window (2018-07-01 to 2019-12-31)
        dt_str = date_obj.strftime('%Y-%m-%d')
        if '2018-07-01' <= dt_str <= '2019-12-31':
            holiday_list.append({
                'Date': dt_str,
                'Day_Name': date_obj.strftime('%A'),
                'Holiday_Name': name,
                'Jurisdiction': 'Thailand (Bank of Thailand)'
            })
    holidays_df = pd.DataFrame(holiday_list)
except ImportError:
    # Fallback to existing CSV if package not installed in active environment
    fallback_path = os.path.join(data_dir, "thai_holidays_18months.csv")
    holidays_df = pd.read_csv(fallback_path)

# Persist to ../data
out_holiday_path = os.path.join(data_dir, "thai_holidays_18months.csv")
holidays_df.to_csv(out_holiday_path, index=False)

print(f"[EVIDENCE SAVED] {len(holidays_df)} Public Holidays saved to: {out_holiday_path}")
display(holidays_df.head(15))

[EVIDENCE SAVED] 36 Public Holidays saved to: c:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\data\thai_holidays_18months.csv


,Date,Day_Name,Holiday_Name,Jurisdiction
0,2018-07-27,Friday,วันอาสาฬหบูชา,Thailand (Bank of Thailand)
1,2018-07-28,Saturday,วันเข้าพรรษา; วันเฉลิมพระชนมพรรษาพระบาทสมเด็จพ...,Thailand (Bank of Thailand)
2,2018-07-30,Monday,ชดเชยวันเข้าพรรษา; ชดเชยวันเฉลิมพระชนมพรรษาพระ...,Thailand (Bank of Thailand)
3,2018-08-12,Sunday,วันเฉลิมพระชนมพรรษาสมเด็จพระบรมราชชนนีพันปีหลว...,Thailand (Bank of Thailand)
4,2018-08-13,Monday,ชดเชยวันเฉลิมพระชนมพรรษาสมเด็จพระบรมราชชนนีพัน...,Thailand (Bank of Thailand)
5,2018-10-13,Saturday,วันคล้ายวันสวรรคตพระบาทสมเด็จพระปรมินทรมหาภูมิ...,Thailand (Bank of Thailand)
6,2018-10-15,Monday,ชดเชยวันคล้ายวันสวรรคตพระบาทสมเด็จพระปรมินทรมห...,Thailand (Bank of Thailand)
7,2018-10-23,Tuesday,วันปิยมหาราช,Thailand (Bank of Thailand)
8,2018-12-05,Wednesday,วันคล้ายวันเฉลิมพระชนมพรรษาพระบาทสมเด็จพระปรมิ...,Thailand (Bank of Thailand)
9,2018-12-10,Monday,วันรัฐธรรมนูญ,Thailand (Bank of Thailand)


## 3. Chulalongkorn University Academic Calendar Ingestion
We record the academic semesters and examination periods for the 2018–2019 academic cycle.

In [3]:
# ==============================================================================
# 3. CHULALONGKORN UNIVERSITY ACADEMIC REGIMES (2018 - 2019)
# ==============================================================================
academic_periods = [
    {'Period_Name': 'First Semester 2018', 'Start_Date': '2018-08-13', 'End_Date': '2018-12-21', 'Regime': 'Regular Academic'},
    {'Period_Name': 'Midterm Exams 2018/1', 'Start_Date': '2018-10-01', 'End_Date': '2018-10-05', 'Regime': 'High Occupancy'},
    {'Period_Name': 'Final Exams 2018/1', 'Start_Date': '2018-12-03', 'End_Date': '2018-12-21', 'Regime': 'Extended Study'},
    {'Period_Name': 'Semester Break 2018/1', 'Start_Date': '2018-12-22', 'End_Date': '2019-01-13', 'Regime': 'Vacation / Recess'},
    {'Period_Name': 'Second Semester 2018', 'Start_Date': '2019-01-14', 'End_Date': '2019-05-24', 'Regime': 'Regular Academic'},
    {'Period_Name': 'Midterm Exams 2018/2', 'Start_Date': '2019-03-04', 'End_Date': '2019-03-08', 'Regime': 'High Occupancy'},
    {'Period_Name': 'Final Exams 2018/2', 'Start_Date': '2019-05-07', 'End_Date': '2019-05-24', 'Regime': 'Extended Study'},
    {'Period_Name': 'Summer Vacation 2019', 'Start_Date': '2019-05-25', 'End_Date': '2019-08-11', 'Regime': 'Low Occupancy'},
    {'Period_Name': 'First Semester 2019', 'Start_Date': '2019-08-12', 'End_Date': '2019-12-20', 'Regime': 'Regular Academic'},
    {'Period_Name': 'Midterm Exams 2019/1', 'Start_Date': '2019-09-30', 'End_Date': '2019-10-04', 'Regime': 'High Occupancy'},
    {'Period_Name': 'Final Exams 2019/1', 'Start_Date': '2019-12-02', 'End_Date': '2019-12-20', 'Regime': 'Extended Study'}
]

academic_df = pd.DataFrame(academic_periods)
out_calendar_path = os.path.join(data_dir, "chula_academic_calendar_18months.csv")
academic_df.to_csv(out_calendar_path, index=False)

print(f"[EVIDENCE SAVED] Chula Academic Calendar saved to: {out_calendar_path}")
display(academic_df)

[EVIDENCE SAVED] Chula Academic Calendar saved to: c:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\data\chula_academic_calendar_18months.csv


,Period_Name,Start_Date,End_Date,Regime
0,First Semester 2018,2018-08-13,2018-12-21,Regular Academic
1,Midterm Exams 2018/1,2018-10-01,2018-10-05,High Occupancy
2,Final Exams 2018/1,2018-12-03,2018-12-21,Extended Study
3,Semester Break 2018/1,2018-12-22,2019-01-13,Vacation / Recess
4,Second Semester 2018,2019-01-14,2019-05-24,Regular Academic
5,Midterm Exams 2018/2,2019-03-04,2019-03-08,High Occupancy
6,Final Exams 2018/2,2019-05-07,2019-05-24,Extended Study
7,Summer Vacation 2019,2019-05-25,2019-08-11,Low Occupancy
8,First Semester 2019,2019-08-12,2019-12-20,Regular Academic
9,Midterm Exams 2019/1,2019-09-30,2019-10-04,High Occupancy


## 4. Verification & Summary
Both external reference datasets have been validated and saved into `../data/`.
Proceed to [`qn1b_raw_ingestion.ipynb`](qn1b_raw_ingestion.ipynb) to ingest the building's physical telemetry.